# 05 — Electoral Dynamics
## Tamil Nadu Assembly Elections (1971–2021)

**Objectives:**
- Anti-incumbency quantification (vote share delta)
- Wave election detection & quantification
- Margin distribution trends
- Voter turnout drivers (correlation analysis)
- Deposit loss / frivolous candidacy trends

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

from db import query, query_all, GENERAL_ELECTION_YEARS, POST_DELIM_YEARS

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

df = query_all()
ge = df[df.year.isin(GENERAL_ELECTION_YEARS)].copy()
winners = ge[ge.position == 1].copy()
print(f'General election records: {len(ge):,}')

## 1. Anti-Incumbency Quantification

In [ ]:
# Calculate ruling party seat change between consecutive elections
election_pairs = list(zip(GENERAL_ELECTION_YEARS[:-1], GENERAL_ELECTION_YEARS[1:]))

anti_inc = []
for prev_yr, curr_yr in election_pairs:
    prev_winners = winners[winners.year == prev_yr]
    curr_winners = winners[winners.year == curr_yr]
    
    if len(prev_winners) == 0 or len(curr_winners) == 0:
        continue
    
    # Ruling party = party with most seats in prev election
    ruling_party = prev_winners.party.value_counts().index[0]
    prev_seats = (prev_winners.party == ruling_party).sum()
    curr_seats = (curr_winners.party == ruling_party).sum()
    total_prev = len(prev_winners)
    total_curr = len(curr_winners)
    
    anti_inc.append({
        'previous': prev_yr, 'current': curr_yr,
        'ruling_party': ruling_party,
        'prev_seats': prev_seats, 'curr_seats': curr_seats,
        'seat_change': curr_seats - prev_seats,
        'prev_share': prev_seats / total_prev * 100,
        'curr_share': curr_seats / total_curr * 100,
        'share_change': (curr_seats / total_curr - prev_seats / total_prev) * 100
    })

ai_df = pd.DataFrame(anti_inc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Seat change
colors = ['#E53935' if x < 0 else '#4CAF50' for x in ai_df.seat_change]
bars = ax1.bar(range(len(ai_df)), ai_df.seat_change, color=colors, edgecolor='white')
ax1.set_xticks(range(len(ai_df)))
ax1.set_xticklabels([f"{r.ruling_party}\n{r.previous}→{r.current}" for _, r in ai_df.iterrows()], fontsize=8)
ax1.set_title('Ruling Party Seat Change (Election to Election)', fontsize=13)
ax1.set_ylabel('Seat Change')
ax1.axhline(y=0, color='black', linewidth=0.5)
for i, row in ai_df.iterrows():
    ax1.annotate(f'{row.seat_change:+.0f}', (i, row.seat_change),
                textcoords='offset points', xytext=(0, 5 if row.seat_change >= 0 else -12), ha='center', fontsize=9)

# Seat share change
colors2 = ['#E53935' if x < 0 else '#4CAF50' for x in ai_df.share_change]
ax2.bar(range(len(ai_df)), ai_df.share_change, color=colors2, edgecolor='white')
ax2.set_xticks(range(len(ai_df)))
ax2.set_xticklabels([f"{r.ruling_party}\n{r.previous}→{r.current}" for _, r in ai_df.iterrows()], fontsize=8)
ax2.set_title('Ruling Party Seat Share Change (%)', fontsize=13)
ax2.set_ylabel('Seat Share Change (%)')
ax2.axhline(y=0, color='black', linewidth=0.5)

plt.suptitle('Anti-Incumbency Effect in Tamil Nadu', fontsize=14)
plt.tight_layout()
plt.savefig('/app/output/05_anti_incumbency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nAnti-incumbency occurred in {(ai_df.seat_change < 0).sum()}/{len(ai_df)} transitions')
print(f'Average seat loss for ruling party: {ai_df.seat_change.mean():.1f} seats')

## 2. Wave Election Detection

In [ ]:
# Define wave: winning party gets >60% of seats
wave_data = []
for year in GENERAL_ELECTION_YEARS:
    yr_winners = winners[winners.year == year]
    total = len(yr_winners)
    if total == 0: continue
    top_party = yr_winners.party.value_counts().index[0]
    top_seats = yr_winners.party.value_counts().values[0]
    top_share = top_seats / total * 100
    wave_data.append({
        'year': year, 'winning_party': top_party,
        'seats': top_seats, 'total': total,
        'seat_share': top_share,
        'is_wave': top_share > 60
    })

wave_df = pd.DataFrame(wave_data)

fig, ax = plt.subplots(figsize=(16, 7))
party_colors = {'DMK': '#E53935', 'ADMK': '#43A047', 'ADK': '#8E24AA', 'INC': '#1E88E5'}
colors = [party_colors.get(p, '#9E9E9E') for p in wave_df.winning_party]
bars = ax.bar(wave_df.year, wave_df.seat_share, color=colors, edgecolor='white', width=3)
ax.axhline(y=60, color='red', linestyle='--', alpha=0.7, label='Wave threshold (60%)')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Simple majority')
ax.set_title('Winning Party Seat Share per Election — Wave Detection', fontsize=14)
ax.set_xlabel('Election Year')
ax.set_ylabel('Seat Share (%)')
ax.legend()

for _, row in wave_df.iterrows():
    label = f"{row.winning_party}\n{row.seat_share:.0f}%"
    style = 'bold' if row.is_wave else 'normal'
    ax.annotate(label, (row.year, row.seat_share), textcoords='offset points',
               xytext=(0, 5), ha='center', fontsize=8, fontweight=style)

plt.tight_layout()
plt.savefig('/app/output/05_wave_elections.png', dpi=150, bbox_inches='tight')
plt.show()

print('Wave elections (>60% seat share):')
print(wave_df[wave_df.is_wave][['year', 'winning_party', 'seats', 'seat_share']].to_string(index=False))

## 3. Margin Distribution Trends

In [ ]:
# Margin distribution by election year
margin_bands = []
for year in GENERAL_ELECTION_YEARS:
    yr_winners = winners[winners.year == year]
    total = len(yr_winners)
    if total == 0: continue
    
    for threshold, label in [(5, '<5%'), (10, '5-10%'), (20, '10-20%')]:
        if threshold == 5:
            count = (yr_winners.margin_percentage < 5).sum()
        elif threshold == 10:
            count = ((yr_winners.margin_percentage >= 5) & (yr_winners.margin_percentage < 10)).sum()
        else:
            count = ((yr_winners.margin_percentage >= 10) & (yr_winners.margin_percentage < 20)).sum()
        margin_bands.append({'year': year, 'band': label, 'count': count, 'pct': count/total*100})
    
    safe = (yr_winners.margin_percentage >= 20).sum()
    margin_bands.append({'year': year, 'band': '≥20%', 'count': safe, 'pct': safe/total*100})

mb_df = pd.DataFrame(margin_bands)

fig, ax = plt.subplots(figsize=(16, 7))
mb_pivot = mb_df.pivot_table(index='year', columns='band', values='pct', fill_value=0)
band_order = ['<5%', '5-10%', '10-20%', '≥20%']
mb_pivot = mb_pivot.reindex(columns=[c for c in band_order if c in mb_pivot.columns])
mb_pivot.plot(kind='bar', stacked=True, ax=ax,
              color=['#E53935', '#FF9800', '#FFC107', '#4CAF50'], edgecolor='white')
ax.set_title('Winner Margin Band Distribution Over Elections', fontsize=14)
ax.set_xlabel('Election Year')
ax.set_ylabel('% of Seats')
ax.legend(title='Margin Band')
plt.tight_layout()
plt.savefig('/app/output/05_margin_bands.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Violin plot of margin % by year (post-delimitation)
post_winners_plot = winners[winners.year.isin(POST_DELIM_YEARS)]
fig = px.violin(post_winners_plot, x='year', y='margin_percentage', box=True, points='all',
                title='Winner Margin Distribution (Post-Delimitation)',
                labels={'margin_percentage': 'Margin (%)', 'year': 'Election Year'})
fig.update_layout(height=500)
fig.show()

## 4. Voter Turnout Drivers

In [ ]:
# Correlation analysis — what drives turnout?
# Use constituency-level aggregates for post-delimitation data
post_const = ge[ge.year.isin(POST_DELIM_YEARS)].drop_duplicates(subset=['year', 'constituency_no'])

corr_vars = ['turnout_percentage', 'n_cand', 'enop', 'electors']
corr_data = post_const[corr_vars].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (x_var, title) in zip(axes, [
    ('n_cand', 'Turnout vs Number of Candidates'),
    ('enop', 'Turnout vs ENOP'),
    ('electors', 'Turnout vs Electorate Size')
]):
    ax.scatter(corr_data[x_var], corr_data.turnout_percentage, alpha=0.3, s=20, color='#2196F3')
    # Add regression line
    slope, intercept, r, p, se = stats.linregress(corr_data[x_var], corr_data.turnout_percentage)
    x_line = np.linspace(corr_data[x_var].min(), corr_data[x_var].max(), 100)
    ax.plot(x_line, intercept + slope * x_line, 'r-', linewidth=2, label=f'r={r:.3f}, p={p:.1e}')
    ax.set_title(title)
    ax.set_xlabel(x_var)
    ax.set_ylabel('Turnout (%)')
    ax.legend()

plt.tight_layout()
plt.savefig('/app/output/05_turnout_drivers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Turnout by constituency type and region
turnout_region = post_const.groupby(['year', 'sub_region']).agg(
    avg_turnout=('turnout_percentage', 'mean')
).reset_index().dropna()

fig = px.line(turnout_region, x='year', y='avg_turnout', color='sub_region',
              markers=True, title='Turnout by Region (Post-Delimitation)',
              labels={'avg_turnout': 'Avg Turnout (%)', 'year': 'Election Year'})
fig.update_layout(height=500)
fig.show()

## 5. Deposit Loss / Frivolous Candidacy

In [ ]:
# Deposit loss percentage per election
dep_data = ge[ge.deposit_lost.isin(['yes', 'no'])].copy()
dep_by_year = dep_data.groupby('year').agg(
    total=('candidate', 'count'),
    lost=('deposit_lost', lambda x: (x == 'yes').sum())
).reset_index()
dep_by_year['lost_pct'] = dep_by_year.lost / dep_by_year.total * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.bar(dep_by_year.year, dep_by_year.lost_pct, color='#FF7043', edgecolor='white')
ax1.set_title('Percentage of Candidates Losing Deposit', fontsize=13)
ax1.set_xlabel('Election Year')
ax1.set_ylabel('Deposit Lost (%)')
for _, row in dep_by_year.iterrows():
    ax1.annotate(f'{row.lost_pct:.0f}%', (row.year, row.lost_pct),
                textcoords='offset points', xytext=(0, 5), ha='center', fontsize=9)

ax2.bar(dep_by_year.year, dep_by_year.lost, color='#AB47BC', edgecolor='white')
ax2.set_title('Absolute Count of Deposit Losses', fontsize=13)
ax2.set_xlabel('Election Year')
ax2.set_ylabel('Count')

plt.suptitle('Frivolous Candidacy Trend', fontsize=14)
plt.tight_layout()
plt.savefig('/app/output/05_deposit_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Average deposit loss rate: {dep_by_year.lost_pct.mean():.1f}%')

In [ ]:
# Party-wise deposit loss (which parties have the most frivolous candidates?)
recent_dep = dep_data[dep_data.year >= 2011]
party_dep = recent_dep.groupby('party').agg(
    total=('candidate', 'count'),
    lost=('deposit_lost', lambda x: (x == 'yes').sum())
).reset_index()
party_dep['lost_pct'] = party_dep.lost / party_dep.total * 100
party_dep = party_dep[party_dep.total >= 20].sort_values('lost_pct')

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlGn_r(party_dep.lost_pct / 100)
ax.barh(party_dep.party, party_dep.lost_pct, color=colors, edgecolor='white')
ax.set_title('Deposit Loss Rate by Party (2011–2021, parties with ≥20 candidates)')
ax.set_xlabel('Deposit Lost (%)')
ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/app/output/05_party_deposit_loss.png', dpi=150, bbox_inches='tight')
plt.show()